# 05 – Validation and Verdicts (R8)

In this notebook we:
1. Compare all models from R7 (from the MLflow run table) and choose one
2. Train the chosen model on all training students
3. Score it **once** on the test set (20% of students we never used before)
4. Check how good its predicted probabilities are
5. Write the final verdicts for H1 and H2

**Rule for choosing the model (set before looking at the test set):** pick the model
with the highest 5-fold CV AUC, as long as its lead over the next model is bigger than
the CV spread (SD). If two models are too close to tell apart, pick the simpler one.

## Step 1 – Load the data, the split and the run table

In [1]:
# Let the notebook find the src folder (one level up)
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score, brier_score_loss
from src.split import load_features, get_train_test

# Same data and same split as every other notebook
df = load_features()
train, test = get_train_test(df)

# The run table exported from MLflow in R7
runs = pd.read_csv("../results/mlflow_runs.csv")

print("Train rows:", len(train), "| Test rows:", len(test))
print("Runs in the table:", len(runs))

Train rows: 95991 | Test rows: 24009
Runs in the table: 6


## Step 2 – Compare the models and choose one
We use the cross-validation results from R7 only. The test set is not used to choose.

We also compute a simple **baseline Brier score**: the score we would get by giving
every student the same chance of failing (the training failure rate). A useful model
should beat it.

In [2]:
# Keep only the columns we need, with short names
table = runs.rename(columns={"tags.mlflow.runName": "model",
                             "metrics.cv_auc_mean": "cv_auc",
                             "metrics.cv_auc_sd": "cv_auc_sd",
                             "metrics.brier_mean": "brier",
                             "metrics.gap": "train_cv_gap"})
table = table[["model", "cv_auc", "cv_auc_sd", "brier", "train_cv_gap"]]
table = table.sort_values("model").reset_index(drop=True)
print(table.round(4).to_string(index=False))

# Baseline: give everyone the training failure rate
rate = train["failed_t3"].mean()
baseline_brier = rate * (1 - rate)
print(f"\nBaseline Brier (everyone gets {rate:.1%}): {baseline_brier:.4f}")

# Apply our rule: highest CV AUC, and is the lead bigger than the SD?
ranked = table.sort_values("cv_auc", ascending=False).reset_index(drop=True)
best, second = ranked.iloc[0], ranked.iloc[1]
lead = best["cv_auc"] - second["cv_auc"]
print(f"\nBest CV AUC: {best['model']} ({best['cv_auc']:.4f} ± {best['cv_auc_sd']:.4f})")
print(f"Next best:   {second['model']} ({second['cv_auc']:.4f})")
print(f"Lead: {lead:.4f}, which is {lead / best['cv_auc_sd']:.1f} times the SD")

                model  cv_auc  cv_auc_sd  brier  train_cv_gap
              M0_base  0.7855     0.0038 0.1187        0.0000
          M1_referral  0.7879     0.0043 0.1195        0.0000
               M2_gpa  0.8735     0.0014 0.1104        0.0005
      M3_gpa_referral  0.8648     0.0017 0.1163        0.0005
   M4_plus_attendance  0.8509     0.0019 0.1507        0.0005
M5_attendance_no_flag  0.8595     0.0019 0.1300        0.0005

Baseline Brier (everyone gets 10.3%): 0.0926

Best CV AUC: M2_gpa (0.8735 ± 0.0014)
Next best:   M3_gpa_referral (0.8648)
Lead: 0.0086, which is 6.0 times the SD


**Result:** M2 (background + term 1 + prior GPA) has the highest CV AUC: 0.874 ± 0.001.
Its lead over the next model (M3, 0.865) is 0.009, which is about 6 times the CV spread,
so the difference is real and not just noise. M2 also has the best CV Brier score (0.110).

**Chosen model: M2.**

All models have a Brier score worse than the simple baseline (0.093, giving everyone
the training failure rate of 10.3%). We check why in Step 5.

## Step 3 – Train the chosen model on all training students
We use the same features and the same preparation as R7. Missing prior GPA is filled
with the median from the **training** students only, so the test set does not
influence anything before it is scored.

In [3]:
# Same features as M2 in R7
BASE = ["failed_t1", "bus_rider", "econ_disadvantaged", "english_learner", "iep",
        "subj_English Language Arts", "subj_Mathematics", "subj_Science"]
GPA = ["prior_year_gpa", "gpa_missing"]
FEATURES = BASE + GPA

def prepare(data):
    """Same preparation as R7: subject -> 0/1 columns, True/False -> 1/0."""
    X = pd.get_dummies(data, columns=["subject_area"], prefix="subj", dtype=int)
    for col in X.select_dtypes("bool").columns:
        X[col] = X[col].astype(int)
    return X

train_X = prepare(train)
test_X = prepare(test)

# Fill missing GPA with the TRAINING median (no information from the test set)
med = train_X["prior_year_gpa"].median()
train_X["prior_year_gpa"] = train_X["prior_year_gpa"].fillna(med)
test_X["prior_year_gpa"] = test_X["prior_year_gpa"].fillna(med)
print("Training median GPA used for missing values:", med)

# Train M2 on all training students
model = GaussianNB().fit(train_X[FEATURES], train_X["failed_t3"])
print("Model trained on", len(train_X), "rows")

Training median GPA used for missing values: 2.83
Model trained on 95991 rows


## Step 4 – Score the test set (once)
This is the only place the test set is used. We report the same scores as R7 so we
can compare: AUC and Brier. If the test AUC is close to the CV AUC, the CV results
from R7 can be trusted.

In [4]:
# Predicted chance of failing for each test row
p_test = model.predict_proba(test_X[FEATURES])[:, 1]
y_test = test_X["failed_t3"]

test_auc = roc_auc_score(y_test, p_test)
test_brier = brier_score_loss(y_test, p_test)

# Baseline on the test set: everyone gets the TRAINING failure rate
test_baseline = brier_score_loss(y_test, np.full(len(y_test), rate))

print(f"Test AUC:   {test_auc:.4f}   (CV AUC in R7: {best['cv_auc']:.4f})")
print(f"Test Brier: {test_brier:.4f}   (CV Brier in R7: {best['brier']:.4f})")
print(f"Baseline Brier on test: {test_baseline:.4f}")

Test AUC:   0.8710   (CV AUC in R7: 0.8735)
Test Brier: 0.1093   (CV Brier in R7: 0.1104)
Baseline Brier on test: 0.0875


**Result:** On the test set, M2 has AUC **0.871**, almost the same as its CV AUC (0.874).
The test Brier score (0.109) is also close to CV (0.110). Since the test students were
never used before, this shows our cross-validation results were trustworthy and the
model is not overfitting.

But the Brier score is still worse than the baseline on the test set (0.088).

## Step 5 – How good are the predicted probabilities?
AUC tells us if the model **ranks** students well (higher risk for those who fail).
Brier also checks if the **probabilities** themselves are right. To see why Brier is
high, we group the test rows by predicted chance and compare it with how many really
failed.

We also check something useful for the school's decision: if the school could only
reach the 20% of students with the highest predicted risk, how many of the real
failures would it catch?

In [5]:
# Compare predicted chance vs. real failure rate in groups
cal = pd.DataFrame({"predicted": p_test, "failed": y_test})
cal["group"] = pd.cut(cal["predicted"], bins=[0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
                      include_lowest=True)
check = cal.groupby("group", observed=True).agg(rows=("failed", "size"),
                                                 avg_predicted=("predicted", "mean"),
                                                 actually_failed=("failed", "mean"))
print(check.to_string(formatters={"avg_predicted": "{:.1%}".format,
                                  "actually_failed": "{:.1%}".format}))

# Top 20% highest predicted risk: how many real failures do they include?
top = cal["predicted"] >= cal["predicted"].quantile(0.80)
caught = cal.loc[top, "failed"].sum() / cal["failed"].sum()
print(f"\nTop 20% highest-risk rows include {caught:.1%} of all term 3 failures in the test set")

                rows avg_predicted actually_failed
group                                             
(-0.001, 0.1]  19073          1.3%            3.8%
(0.1, 0.3]      1205         17.2%           16.0%
(0.3, 0.5]       356         39.0%           17.1%
(0.5, 0.7]       283         60.7%           11.7%
(0.7, 0.9]       374         82.0%           18.2%
(0.9, 1.0]      2718         98.3%           46.1%

Top 20% highest-risk rows include 68.2% of all term 3 failures in the test set


**Result:** The model **ranks** students well, but its **probabilities are too extreme**:
- When the model says above 90% (2,718 rows), only **46%** actually failed.
- When it says below 10% (19,073 rows), **3.8%** failed, not 1.3%.

This is a known weakness of naive Bayes: it assumes the features are independent, so
when several features point the same way (for example, low GPA and failing term 1), it
multiplies the evidence and becomes too sure. That is why the Brier score is worse than
the baseline even though the AUC is high.

**For the school's decision:** the ranking is still useful. The **top 20%** of rows with
the highest predicted risk include **68% of all term 3 failures** in the test set. So
the model can help decide *who to check on first*, but its numbers should not be read
as exact chances of failing.

In [6]:
# Save the test result so the README can point to it
from pathlib import Path
Path("../results").mkdir(exist_ok=True)
pd.DataFrame([{"model": best["model"], "cv_auc": best["cv_auc"], "test_auc": test_auc,
               "cv_brier": best["brier"], "test_brier": test_brier,
               "baseline_brier": test_baseline, "top20_catches": caught}]
            ).to_csv("../results/test_result.csv", index=False)
print("Saved ../results/test_result.csv")

Saved ../results/test_result.csv


## Step 6 – Final verdicts

| | Planned test (R5) | R6 experiments | R7/R8 models | Verdict |
|---|---|---|---|---|
| **H1** | Compare referred vs. not referred inside groups with the same prior GPA and term 1 result | Referred students failed **less** in all 7 groups, even though they fail more overall (19.6% vs. 8.9%) | Referral adds nothing to prediction once prior GPA is known (M2 0.874 → M3 0.865) | **Supported** |
| **H2** | Failure goes down at every attendance step, inside each GPA group | Planned rule not met (2 of 5 groups); lower-GPA students who attended most failed much less (Q1: 44.3% → 18.7%) | Attendance does not improve prediction (M4 0.851, M5 0.860) | **Revised** |

**What this means:**
- The tutoring paradox is **Simpson's paradox**. Referred students fail more overall only
  because the students who get referred were already struggling. Compared with similar
  students, referred students do better.
- Referral and attendance don't help *predict* failure, because they mostly repeat
  what prior GPA already tells the model. That is a different question from whether
  tutoring *helps*, which is what R6 tested.

**Recommendation:**
1. Don't judge tutoring by comparing raw failure rates of referred and non-referred
   students. Always compare students with similar prior grades.
2. Focus on attendance for struggling students (lower GPA groups), where it made the
   biggest difference, and look at transportation, since bus riders attend 6–9 points less.
3. M2 can be used to rank students for early check-ins (the top 20% catches 68% of
   failures), but its probabilities should be calibrated before being shown as chances.

**Limitations:**
- This is past data, not an experiment, so we cannot prove tutoring *causes* lower
  failure. Motivation and family support are not in the data.
- GaussianNB treats 0/1 features as normal distributions and assumes independence;
  a calibrated or different model could give better probabilities.
- One school year only; results may differ in other years.